In [ ]:
!pip install transformers
!pip install datasets
!pip install accelerate

In [ ]:
import os
import torch

from transformers import AutoModel
from transformers import AutoTokenizer

# For Classification
from transformers import AutoModelForSequenceClassification

# For Fine-Tuning
from datasets import load_dataset
from transformers import Trainer
from transformers import TrainingArguments

# TinyBERT (Jiao et al., 2020)

In [ ]:
tiny_bert_name = 'huawei-noah/TinyBERT_General_4L_312D'

tiny_bert_model = AutoModel.from_pretrained(tiny_bert_name)
tiny_bert_tokenizer = AutoTokenizer.from_pretrained(tiny_bert_name)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
tiny_bert_size_bytes = sum([p.numel() * p.element_size() for p in tiny_bert_model.parameters()])
tiny_bert_num_parameters = sum([p.numel() for p in tiny_bert_model.parameters()])
print(f'TinyBERT Size: {tiny_bert_size_bytes / 1024 / 1024:.4f} MB')
print(f'TinyBERT Num Parameters: {tiny_bert_num_parameters}')

TinyBERT Size: 54.7419 MB
TinyBERT Num Parameters: 14350248


In [ ]:
input_text = 'TinyBERT is very small, but efficient.'
inputs = tiny_bert_tokenizer(input_text, return_tensors='pt')

print(f'Inputs from TinyBERT Tokenizer: \n {inputs}')

with torch.no_grad():
    outputs = tiny_bert_model(**inputs)

print(outputs.last_hidden_state.shape)

Inputs from TinyBERT Tokenizer: 
 {'input_ids': tensor([[ 101,  100, 2003, 2200, 2235, 1010, 2021, 8114, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 10, 312])


## TinyBERT for Classification

In [ ]:
auto_model_sequence_classification = AutoModelForSequenceClassification.from_pretrained(
    tiny_bert_name,
    num_labels=2 # change depending on number of classes
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not o

## Fine-Tuning TinyBERT

In [ ]:
tiny_bert_dataset = load_dataset('glue', 'sst2')

training_args = TrainingArguments(
    output_dir='./tinybert-sst2',
    evaluation_strategy='epoch',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True
)

trainer = Trainer(
    model=tiny_bert_model,
    args=training_args,
    train_dataset=tiny_bert_dataset['train'],
    eval_dataset=tiny_bert_dataset['validation']
)

trainer.train()

# MiniLM (Wang et al., 2020)

In [ ]:
mini_lm_name = 'microsoft/MiniLM-L12-H384-uncased'

mini_lm_model = AutoModel.from_pretrained(mini_lm_name)
mini_lm_tokenizer = AutoTokenizer.from_pretrained(mini_lm_name)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/133M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
mini_lm_size_bytes = sum([p.numel() * p.element_size() for p in mini_lm_model.parameters()])
mini_lm_num_parameters = sum([p.numel() for p in mini_lm_model.parameters()])
print(f'MiniLM Size: {mini_lm_size_bytes / 1024 / 1024:.4f} MB')
print(f'MiniLM Num Parameters: {mini_lm_num_parameters}')

MiniLM Size: 127.2583 MB
MiniLM Num Parameters: 33360000


# DistilBERT (Sanh et al., 2019)

Paper: "DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter"

In [ ]:
distil_bert_name = 'distilbert-base-uncased'

distil_bert_model = AutoModel.from_pretrained(distil_bert_name)
distil_bert_tokenizer = AutoTokenizer.from_pretrained(distil_bert_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
distil_bert_size_bytes = sum([p.numel() * p.element_size() for p in distil_bert_model.parameters()])
distil_bert_num_parameters = sum([p.numel() for p in distil_bert_model.parameters()])
print(f'DistilBERT Size: {distil_bert_size_bytes / 1024 / 1024:.4f} MB')
print(f'DistilBERT Num Parameters: {distil_bert_num_parameters}')

DistilBERT Size: 253.1543 MB
DistilBERT Num Parameters: 66362880


# ALBERT‑base (Lan et al., 2019)

Paper: "ALBERT: A Lite BERT for Self‑supervised Learning of Language Representations"

In [ ]:
albert_base_name = 'albert-base-v2'

albert_base_model = AutoModel.from_pretrained(albert_base_name)
albert_base_tokenizer = AutoTokenizer.from_pretrained(albert_base_name)

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertModel LOAD REPORT from: albert-base-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
albert_base_size_bytes = sum([p.numel() * p.element_size() for p in albert_base_model.parameters()])
albert_base_num_parameters = sum([p.numel() for p in albert_base_model.parameters()])
print(f'ALBERT-base Size: {albert_base_size_bytes / 1024 / 1024:.4f} MB')
print(f'ALBERT-base Num Parameters: {albert_base_num_parameters}')

ALBERT-base Size: 44.5693 MB
ALBERT-base Num Parameters: 11683584


# ELECTRA-base (Clark et al., 2020)

Paper: "ELECTRA: Pre‑training Text Encoders as Discriminators Rather Than Generators"

In [ ]:
electra_base_name = 'google/electra-base-discriminator'

electra_base_model = AutoModel.from_pretrained(electra_base_name)
electra_base_tokenizer = AutoTokenizer.from_pretrained(electra_base_name)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
electra.embeddings_project.bias                   | UNEXPECTED |  | 
electra.embeddings_project.weight                 | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
electra_base_size_bytes = sum([p.numel() * p.element_size() for p in electra_base_model.parameters()])
electra_base_num_parameters = sum([p.numel() for p in electra_base_model.parameters()])
print(f'ELECTRA-base Size: {electra_base_size_bytes / 1024 / 1024:.4f} MB')
print(f'ELECTRA-base Num Parameters: {electra_base_num_parameters}')

ELECTRA-base Size: 415.3887 MB
ELECTRA-base Num Parameters: 108891648


# DeBERTa‑v3 base (He et al., 2021)

Paper: "DeBERTaV3: Improving DeBERTa using ELECTRA‑style pretraining with gradient‑disentangled embedding sharing"

In [ ]:
deberta_v3_base_name = 'microsoft/deberta-v3-base'

deberta_v3_base_model = AutoModel.from_pretrained(deberta_v3_base_name)
deberta_v3_base_tokenizer = AutoTokenizer.from_pretrained(deberta_v3_base_name)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [ ]:
deberta_v3_base_size_bytes = sum([p.numel() * p.element_size() for p in deberta_v3_base_model.parameters()])
deberta_v3_base_num_parameters = sum([p.numel() for p in deberta_v3_base_model.parameters()])
print(f'DeBERTa-v3 base Size: {deberta_v3_base_size_bytes / 1024 / 1024:.4f} MB')
print(f'DeBERTa-v3 base Num Parameters: {deberta_v3_base_num_parameters}')

DeBERTa-v3 base Size: 350.6309 MB
DeBERTa-v3 base Num Parameters: 183831552


# BERT‑medium (Turc et al., 2019)

Paper: "Well‑Read Students Learn Better: On the Importance of Pre‑training Compact Models"

In [ ]:
bert_medium_name = 'google/bert_uncased_L-8_H-512_A-8'

bert_medium_model = AutoModel.from_pretrained(bert_medium_name)
bert_medium_tokenizer = AutoTokenizer.from_pretrained(bert_medium_name)

config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/135 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/bert_uncased_L-8_H-512_A-8
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
bert_medium_size_bytes = sum([p.numel() * p.element_size() for p in bert_medium_model.parameters()])
bert_medium_num_parameters = sum([p.numel() for p in bert_medium_model.parameters()])
print(f'BERT-medium Size: {bert_medium_size_bytes / 1024 / 1024:.4f} MB')
print(f'BERT-medium Num Parameters: {bert_medium_num_parameters}')

BERT-medium Size: 157.8262 MB
BERT-medium Num Parameters: 41373184


# RoBERTa‑base (Liu et al., 2019)

Paper: "RoBERTa: A Robustly Optimized BERT Pretraining Approach"

In [ ]:
roberta_base_name = 'roberta-base'

roberta_base_model = AutoModel.from_pretrained(roberta_base_name)
roberta_base_tokenizer = AutoTokenizer.from_pretrained(roberta_base_name)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
roberta_base_size_bytes = sum([p.numel() * p.element_size() for p in roberta_base_model.parameters()])
roberta_base_num_parameters = sum([p.numel() for p in roberta_base_model.parameters()])
print(f'RoBERTa-base Size: {roberta_base_size_bytes / 1024 / 1024:.4f} MB')
print(f'RoBERTa-base Num Parameters: {roberta_base_num_parameters}')

RoBERTa-base Size: 475.4854 MB
RoBERTa-base Num Parameters: 124645632


# T5‑Large (Raffel et al., 2020)

Paper: "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"

In [ ]:
t5_large_name = 't5-large'

t5_large_model = AutoModel.from_pretrained(t5_large_name)
t5_large_tokenizer = AutoTokenizer.from_pretrained(t5_large_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
t5_large_size_bytes = sum([p.numel() * p.element_size() for p in t5_large_model.parameters()])
t5_large_num_parameters = sum([p.numel() for p in t5_large_model.parameters()])
print(f'T5-Large Size: {t5_large_size_bytes / 1024 / 1024:.4f} MB')
print(f'T5-Large Num Parameters: {t5_large_num_parameters}')

T5-Large Size: 2813.9805 MB
T5-Large Num Parameters: 737668096


# DeBERTa‑v2 XLarge (He et al., 2020)

Paper: "DeBERTa: Decoding-enhanced BERT with Disentangled Attention"

In [ ]:
deberta_v2_xlarge_name = 'microsoft/deberta-xlarge'

deberta_v2_xlarge_model = AutoModel.from_pretrained(deberta_v2_xlarge_name)
deberta_v2_xlarge_tokenizer = AutoTokenizer.from_pretrained(deberta_v2_xlarge_name)

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/772 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-xlarge
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [ ]:
deberta_v2_xlarge_size_bytes = sum([p.numel() * p.element_size() for p in deberta_v2_xlarge_model.parameters()])
deberta_v2_xlarge_num_parameters = sum([p.numel() for p in deberta_v2_xlarge_model.parameters()])
print(f'DeBERTa-v2 XLarge Size: {deberta_v2_xlarge_size_bytes / 1024 / 1024:.4f} MB')
print(f'DeBERTa-v2 XLarge Num Parameters: {deberta_v2_xlarge_num_parameters}')

DeBERTa-v2 XLarge Size: 1445.5840 MB
DeBERTa-v2 XLarge Num Parameters: 757804032


# DeBERTa‑v3 Large (He et al., 2021)

Paper: "DeBERTaV3: Improving DeBERTa using ELECTRA-style pretraining"

In [ ]:
deberta_v3_large_name = 'microsoft/deberta-v3-large'

deberta_v3_large_model = AutoModel.from_pretrained(deberta_v3_large_name)
deberta_v3_large_tokenizer = AutoTokenizer.from_pretrained(deberta_v3_large_name)

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [ ]:
deberta_v3_large_size_bytes = sum([p.numel() * p.element_size() for p in deberta_v3_large_model.parameters()])
deberta_v3_large_num_parameters = sum([p.numel() for p in deberta_v3_large_model.parameters()])
print(f'DeBERTa-v3 Large Size: {deberta_v3_large_size_bytes / 1024 / 1024:.4f} MB')
print(f'DeBERTa-v3 Large Num Parameters: {deberta_v3_large_num_parameters}')

DeBERTa-v3 Large Size: 827.8125 MB
DeBERTa-v3 Large Num Parameters: 434012160


# T5‑3B (Raffel et al., 2020)

Paper: "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"

In [ ]:
t5_3b_name = 't5-3b'

t5_3b_model = AutoModel.from_pretrained(t5_3b_name)
t5_3b_tokenizer = AutoTokenizer.from_pretrained(t5_3b_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/11.4G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
t5_3b_size_bytes = sum([p.numel() * p.element_size() for p in t5_3b_model.parameters()])
t5_3b_num_parameters = sum([p.numel() for p in t5_3b_model.parameters()])
print(f'T5-3B Size: {t5_3b_size_bytes / 1024 / 1024:.4f} MB')
print(f'T5-3B Num Parameters: {t5_3b_num_parameters}')

T5-3B Size: 10877.9844 MB
T5-3B Num Parameters: 2851598336


# RoBERTa‑Large (Liu et al., 2019)

Paper: "RoBERTa: A Robustly Optimized BERT Pretraining Approach"

In [ ]:
roberta_large_name = 'roberta-large'

roberta_large_model = AutoModel.from_pretrained(roberta_large_name)
roberta_large_tokenizer = AutoTokenizer.from_pretrained(roberta_large_name)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
roberta_large_size_bytes = sum([p.numel() * p.element_size() for p in roberta_large_model.parameters()])
roberta_large_num_parameters = sum([p.numel() for p in roberta_large_model.parameters()])
print(f'RoBERTa-Large Size: {roberta_large_size_bytes / 1024 / 1024:.4f} MB')
print(f'RoBERTa-Large Num Parameters: {roberta_large_num_parameters}')

RoBERTa-Large Size: 1355.5898 MB
RoBERTa-Large Num Parameters: 355359744


# BERT‑Large (Whole Word Masking) (Devlin et al., 2018)

Paper: "BERT: Pre-training of Deep Bidirectional Transformers"

In [ ]:
bert_large_name = 'bert-large-uncased-whole-word-masking'

bert_large_model = AutoModel.from_pretrained(bert_large_name)
bert_large_tokenizer = AutoTokenizer.from_pretrained(bert_large_name)

config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-large-uncased-whole-word-masking
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
bert_large_size_bytes = sum([p.numel() * p.element_size() for p in bert_large_model.parameters()])
bert_large_num_parameters = sum([p.numel() for p in bert_large_model.parameters()])
print(f'BERT-Large Size: {bert_large_size_bytes / 1024 / 1024:.4f} MB')
print(f'BERT-Large Num Parameters: {bert_large_num_parameters}')

BERT-Large Size: 1278.4648 MB
BERT-Large Num Parameters: 335141888


# Datasets

## GLUE Benchmark (Wang et al., 2018)

Paper: "GLUE: A Multi-Task Benchmark and Analysis Platform for Natural Language Understanding"

In [ ]:
sst2 = load_dataset('glue', 'sst2')
mnli = load_dataset('glue', 'mnli')
cola = load_dataset('glue', 'cola')

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1063 [00:00<?, ? examples/s]

In [ ]:
print(sst2['train'])
print(sst2['test'])

print(mnli['train'])
# print(mnli['test']) # mnli does not have test

print(cola['train'])
print(cola['test'])

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 67349
})
Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 1821
})
Dataset({
    features: ['premise', 'hypothesis', 'label', 'idx'],
    num_rows: 392702
})
Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 8551
})
Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 1063
})


Papers for Ablation Studies:

T5-3B:

https://arxiv.org/abs/1910.10683

Model: Introduces T5 with sizes up to 11B, including T5-3B as one of the main checkpoints.

Ablations:

Pretraining objectives (span corruption variants).

Model size scaling (60M → 11B, including 3B).

Dataset variants (C4 filtering, etc.).

Transfer strategies and multi-task vs pretrain-then-finetune.

Where to look:

Ablation-style results are spread across the experimental sections and appendices (e.g., comparisons of objectives, data, and scale).

The “Discussion” and “Conclusion” sections implicitly point to open questions about scaling, data quality, and objective design.

How you can build on it (ablation-focused):

Revisit scaling laws specifically at 3B:

Reproduce a subset of their experiments but densely sample around the 3B regime (e.g., 1B, 2B, 3B, 4B) to see whether 3B is actually a “sweet spot” for certain tasks vs others.

Objective ablations at fixed 3B:

Keep architecture and data fixed, but systematically vary:

corruption rate, span length distribution, masking strategy;

mixing ratio of unsupervised vs supervised tasks.

Many of these are only coarsely explored in the original paper—there’s room for a more fine-grained, 3B-specific study.

https://arxiv.org/abs/2403.08970


Model: Uses a T5-3B model for pseudo-positive labeling in dense retrieval domain adaptation.

Ablations:

Different pseudo-labeling strategies and negative sampling choices.

Future work hints:

The way T5-3B is used as a labeler is relatively fixed—there’s room to ask: how does changing the T5-3B configuration itself affect the quality of pseudo-labels?

How you can build on it (original twist):

Ablate T5-3B “labeler” variants:

Compare: base T5-3B vs domain-continued-pretrained T5-3B vs instruction-tuned T5-3B as pseudo-labelers.

Evaluate downstream retrieval performance and label quality metrics (agreement, calibration).

Prompting vs fine-tuning ablation:

Use T5-3B either:

zero-shot/few-shot prompted;

lightly fine-tuned for labeling;

heavily fine-tuned.

Ablate how much extra training is actually worth it for pseudo-labeling quality.







## Ideas

We test for T5-large

**“Where does T5-3B actually need capacity?”** (large)
- [Parameter-Efficient Fine-Tuning Design Spaces](https://arxiv.org/pdf/2301.01821)
- [Exploring Selective Layer Freezing Strategies](https://www.mdpi.com/2076-3417/15/19/10434)
- [The Original T5 Paper](https://arxiv.org/pdf/1910.10683)

**“Adapters vs continued pretraining for domain shift at 3B”** (large)
- [Don't Stop Pretraining](https://arxiv.org/abs/2004.10964)
- [How Useful is Continued Pre-Training for Generative Unsupervised Domain Adaptation?](https://aclanthology.org/2024.repl4nlp-1.9.pdf)
- [Continual Pre-Training is (not) What You Need" (2025)](https://arxiv.org/pdf/2504.13603)
- [Multitask Prompted Training Enables Zero-Shot Task Generalization](https://arxiv.org/pdf/2110.08207)
- [The FLAN Paper (Chung et al., 2022)](https://arxiv.org/pdf/2210.11416)

Potentially explore some other models, maybe less memory prone.
- Explore T5Gemma-2B or T5Gemma-Large
- Explore LLmama 3.2


<br>
<br>
<br>


**Distillation:**
- https://deepmind.google/models/gemma/gemma-4/
- LoRA (Low-Rank Adaptation) / ELA fine tune
- https://ai.google.dev/gemma/docs/core/model_card_4?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content (E2B model)

**Cross Architecture Distillation**
- each architecture comes with own bias

**Different Methods of Distillation could get different results based on inductive bias of each architecture (class of models)**

Select 2-3 methods of distillation (from Transformer to State-Space Model/GRU/LSTM/CNN) -> ranking between distillation methods on these scenarios (make sure the students (SSM, GRU, etc) have similar number of parameters for clear comparisons).

Distillation Methods:
- DistiLLM-2 (https://arxiv.org/abs/2503.07067)
- DA-KD (https://openreview.net/forum?id=NCYBdRCpw1)
- MiniLLM (https://arxiv.org/abs/2306.08543)



Models:
LLama 8b (might be too big, 8b params, could try lower precision), **QWEN 3.5 (0.8billion - 8billion)**, PHI 4-mini, LLama 3.2 (3b params), Mistral (3b params)


Mistral 3:
https://arxiv.org/pdf/2601.08584

Qwen3:
https://arxiv.org/pdf/2505.09388

<br>
<br>
<br>
<br>


Datasets:
- https://huggingface.co/datasets/databricks/databricks-dolly-15k (use this for teacher inference, the student will learn from the outputs of teacher)



Here are some original, T5-3B-focused ablation ideas that are realistic for an individual researcher:

**“Where does T5-3B actually need capacity?”**

Idea: Freeze most of the model and only fine-tune specific blocks (e.g., last 4 encoder layers, or cross-attention only). (for T5-3B might not enter in memory)

Ablation: Which subset of parameters gives the best performance per trainable-parameter?

Contribution: A detailed map of “capacity-critical” regions in T5-3B.

“Objective granularity at fixed scale (3B)”

Idea: At 3B, systematically vary only the pretraining objective hyperparameters (masking rate, span length, corruption type) on a smaller but controlled pretraining budget.

Ablation: Show which objective variants give the best transfer efficiency per FLOP.

Contribution: Practical guidance for “cheap” re-pretraining of T5-3B for new domains.

**“Adapters vs continued pretraining for domain shift at 3B”**

Idea: Compare two strategies for domain adaptation of T5-3B:

(a) continued pretraining on domain text;

(b) PEFT (LoRA/QLoRA) on downstream tasks only.

Ablation: Vary domain size, domain shift severity, and compute budget.

Contribution: A clear decision chart: when to invest in continued pretraining vs just adapters for a 3B model.

“Prompt format and instruction tuning ablations for T5-3B”

Idea: Take a FLAN-style instruction-tuning setup but fix the backbone to T5-3B and ablate:

prompt templates;

mixture of instruction types;

presence/absence of chain-of-thought.

Contribution: Show how sensitive a 3B encoder–decoder is to prompt design vs data quantity.

https://arxiv.org/abs/2205.05131

1. Objective Ablation at Fixed Scale (3B)
UL2 mixes three denoising objectives.
But the paper does not deeply explore:

how the mixture behaves specifically at 3B,

whether 3B is a “sweet spot” for certain objective ratios,

or whether the mixture is optimal for mid‑sized models.

Your contribution
Run controlled experiments where you vary:

R:S:X ratios

corruption spans

masking rates

noise schedules

All at exactly 3B.

This is a clean, original ablation study.

2. Scaling‑down analysis (20B → 3B)
UL2 shows strong results at 20B, but the 3B model is under‑analyzed.

Your contribution
Study:

which UL2 behaviors survive at 3B,

which degrade,

and why.

This is a scaling‑law‑style ablation, but focused on objective sensitivity, not just model size.

3. Domain‑specific UL2‑3B continued pretraining
UL2 is trained on general web text.
The paper suggests exploring domain‑adapted variants.

Your contribution
Pick a domain:

legal

medical

finance

code

Romanian language (since you’re in Romania)

Then compare:

continued pretraining vs.

PEFT (LoRA/QLoRA) vs.

instruction tuning

All on UL2‑3B.

This is publishable and computationally feasible.

4. Mixture‑of‑Denoisers vs. Instruction Tuning
UL2 predates FLAN.
The paper hints that UL2 might synergize with instruction tuning, but does not test it.

Your contribution
Take UL2‑3B and:

instruction‑tune it (FLAN‑style),

ablate prompt formats,

ablate instruction mixture sizes,

compare to T5‑3B and FLAN‑T5‑3B.

This is a very strong research direction.

5. Efficiency Ablations (Memory, Throughput, Latency)
UL2‑3B is large enough to matter but small enough to experiment with.

Your contribution
Benchmark:

FlashAttention vs. standard attention

BF16 vs. FP16 vs. 8‑bit

LoRA rank ablations

Encoder‑only vs. decoder‑only fine‑tuning

This is practical and valuable for the community.

In [ ]:
raw_datasets = load_dataset("cnn_dailymail", "3.0.0", split={"train": "train[:1%]", "validation": "validation[:1%]"})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
print(raw_datasets['train'])

for i in range(min(10, len(raw_datasets['train']))):
    entry = raw_datasets['train'][i]
    print(entry['article'])
    print(entry['highlights'])

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 2871
})
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently s

bitsandbytes (4-bit loading) and DeepSpeed or FSDP.

https://arxiv.org/pdf/2301.01821

They used T5-base and T5-3B to find the best "patterns" for allocating trainable parameters.

https://www.mdpi.com/2076-3417/15/19/10434

The Original T5 Paper (Raffel et al., 2019):
They did basic ablations on adapter layers and "gradual unfreezing," but because they were introducing the whole framework, they didn't provide the granular "capacity map" you’re proposing.

In [ ]:
gemma_2b_name = 'google/gemma-2b'

gemma_2b_model = AutoModel.from_pretrained(gemma_2b_name)
gemma_2b_tokenizer = AutoTokenizer.from_pretrained(gemma_2b_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2b.
401 Client Error. (Request ID: Root=1-69fcc025-40eaf136221284002e8a3b2a;62e271b2-c15e-4a8c-9853-ab57f506b0ca)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b/resolve/main/config.json.
Access to model google/gemma-2b is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
gemma_2b_size_bytes = sum([p.numel() * p.element_size() for p in gemma_2b_model.parameters()])
gemma_2b_num_parameters = sum([p.numel() for p in gemma_2b_model.parameters()])
print(f'Gemma-2b Size: {gemma_2b_size_bytes / 1024 / 1024:.4f} MB')
print(f'BERT-Large Num Parameters: {gemma_2b_num_parameters}')